<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/AST/Inference_with_the_Audio_Spectogram_Transformer_to_classify_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Set-up environment

First we install 🤗 Transformers.

In [1]:
!pip install -q transformers soundfile


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Load audio

Let's load some audio on which we'd like to test the model. We'll use soundfile to load the audio file.

In [2]:
from huggingface_hub import hf_hub_download
import soundfile as sf
import IPython

filepath = hf_hub_download(
    repo_id="nielsr/audio-spectogram-transformer-checkpoint",
    filename="sample_audio.flac",
    repo_type="dataset"
)

waveform, sampling_rate = sf.read(filepath)
print(f"Sampling rate: {sampling_rate} Hz")
print(f"Audio shape: {waveform.shape}")

IPython.display.Audio(waveform, rate=sampling_rate)

/Users/nielsrogge/Documents/python_projecten/Transformers-Tutorials/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sampling rate: 16000 Hz
Audio shape: (160125,)


## Prepare audio for the model (using feature extractor)

We can prepare the audio using `AutoFeatureExtractor`, which turns it into a spectrogram tensor of shape (batch_size, time_dimension, frequency_dimension). The feature extractor also handles normalization using the mean and standard deviation from the AudioSet dataset.

In [3]:
from transformers import AutoFeatureExtractor

model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

/Users/nielsrogge/Documents/python_projecten/Transformers-Tutorials/.venv/lib/python3.12/site-packages/transformers/audio_utils.py:525: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (257) may be set too low.
  warnings.warn(


In [4]:
inputs = feature_extractor(waveform, sampling_rate=sampling_rate, return_tensors="pt")
print(f"Input values shape: {inputs.input_values.shape}")

Input values shape: torch.Size([1, 1024, 128])


## Load model

Next we load one of the models that the AST authors released from the [hub](https://huggingface.co/models?other=audio-spectrogram-transformer).

This one was fine-tuned on AudioSet, an important benchmark for audio classification.

In [5]:
from transformers import ASTForAudioClassification
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = ASTForAudioClassification.from_pretrained(
    model_id,
    attn_implementation="sdpa",  # Use PyTorch's efficient SDPA for faster inference
).to(device)

## Forward pass

Next let's forward the audio through the model! We perform an argmax on the model's logits to get the predicted class index. We use model.config.id2label to turn that back into text.

In [6]:
inputs = inputs.to(device)

with torch.no_grad():
    outputs = model(**inputs)

In [7]:
predicted_class_idx = outputs.logits.argmax(-1).item()
predicted_label = model.config.id2label[predicted_class_idx]
print(f"Predicted class: {predicted_label}")

Predicted class: Organ


In [8]:
# Show top 5 predictions with probabilities
import torch.nn.functional as F

probs = F.softmax(outputs.logits, dim=-1)
top5_probs, top5_indices = torch.topk(probs, 5)

print("Top 5 predictions:")
for prob, idx in zip(top5_probs[0], top5_indices[0]):
    label = model.config.id2label[idx.item()]
    print(f"  {label}: {prob.item():.2%}")

Top 5 predictions:
  Organ: 56.68%
  Hammond organ: 14.79%
  Keyboard (musical): 7.33%
  Piano: 5.31%
  Speech: 4.41%
